# Audio-only Emotion Baseline (WavLM)

MELD 오디오를 WavLM으로 인코딩해 7개 감정을 분류

## 1. 환경 설정과 재현성

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import AutoFeatureExtractor, WavLMModel
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. 데이터 경로와 유효 샘플 준비

In [ ]:
data_dir = Path("/workspace/emotion-ai/data/meld_audio")
train_audio_dir = data_dir / "extracted/train"
dev_audio_dir = data_dir / "extracted/dev"
test_audio_dir = data_dir / "extracted/test"

train_df = pd.read_csv(data_dir / "train.csv")
dev_df = pd.read_csv(data_dir / "dev.csv")
test_df = pd.read_csv(data_dir / "test.csv")

print("train:", len(train_df))
print("dev  :", len(dev_df))
print("test :", len(test_df))


In [ ]:
# 오디오가 실제 존재하는 train 행
train_df["audio_exists"] = train_df.apply(
    lambda row: (
        train_audio_dir
        / f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.flac"
    ).exists(),
    axis=1,
)

print("missing train audio:", (~train_df["audio_exists"]).sum())
train_df = train_df[train_df["audio_exists"]].reset_index(drop=True)
print("filtered train:", len(train_df))


In [ ]:
# 오디오가 실제 존재하는 dev 행
dev_df["audio_exists"] = dev_df.apply(
    lambda row: (
        dev_audio_dir
        / f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.flac"
    ).exists(),
    axis=1,
)

print(
    dev_df.loc[
        ~dev_df["audio_exists"],
        ["Dialogue_ID", "Utterance_ID", "Utterance", "Emotion"],
    ]
)

dev_df = dev_df[dev_df["audio_exists"]].reset_index(drop=True)
print("filtered dev:", len(dev_df))


In [ ]:
# 손상된 것으로 확인된 test 오디오 2개를 평가 대상 제외
bad_test_audio = {
    "dia38_utt4.flac",
    "dia220_utt0.flac",
}

test_df["audio_file"] = (
    "dia"
    + test_df["Dialogue_ID"].astype(str)
    + "_utt"
    + test_df["Utterance_ID"].astype(str)
    + ".flac"
)

test_df_audio = test_df[
    ~test_df["audio_file"].isin(bad_test_audio)
].reset_index(drop=True)

print("original test:", len(test_df))
print("audio-valid test:", len(test_df_audio))


## 3. Labels와 Feature Extractor

In [11]:
# 7개 감정 라벨과 정수 ID의 대응표를 만듭니다.
label2id = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "joy": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}

id2label = {v: k for k, v in label2id.items()}

In [ ]:
# WavLM 입력 정규화와 padding을 담당하는 feature extractor를 불러옵니다.
MODEL_NAME = "microsoft/wavlm-base-plus"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

## 4. Dataset과 Collator

In [12]:
# 오디오 파일을 mono float tensor와 라벨로 반환하는 Dataset을 정의합니다.
import torch
import soundfile as sf

from torch.utils.data import Dataset
from pathlib import Path


class AudioDataset(Dataset):
    def __init__(
        self,
        dataframe,
        audio_dir,
        label2id,
        target_sample_rate=16000,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.label2id = label2id
        self.target_sample_rate = target_sample_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        dialogue_id = row["Dialogue_ID"]
        utterance_id = row["Utterance_ID"]

        audio_path = (
            self.audio_dir
            / f"dia{dialogue_id}_utt{utterance_id}.flac"
        )

        waveform, sample_rate = sf.read(
            audio_path,
            dtype="float32"
        )

        # 현재 MELD audio가 mono인지 확인
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)

        if sample_rate != self.target_sample_rate:
            raise ValueError(
                f"Unexpected sample rate: "
                f"{sample_rate} at {audio_path}"
            )

        label = self.label2id[row["Emotion"]]

        waveform = torch.from_numpy(waveform)

        return {
            "waveform": waveform,
            "label": torch.tensor(label, dtype=torch.long),
            "audio_path": str(audio_path),
        }

In [19]:
# 길이가 다른 파형을 padding하고 attention mask를 만드는 collator를 정의합니다.
class AudioCollator:
    def __init__(
        self,
        feature_extractor,
        sampling_rate=16000,
    ):
        self.feature_extractor = feature_extractor
        self.sampling_rate = sampling_rate

    def __call__(self, batch):

        waveforms = [
            item["waveform"].numpy()
            for item in batch
        ]

        labels = torch.stack([
            item["label"]
            for item in batch
        ])

        audio_paths = [
            item["audio_path"]
            for item in batch
        ]

        inputs = self.feature_extractor(
            waveforms,
            sampling_rate=self.sampling_rate,
            padding=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_values": inputs["input_values"],
            "attention_mask": inputs["attention_mask"],
            "labels": labels,
            "audio_paths": audio_paths,
        }

In [ ]:
train_dataset = AudioDataset(
    dataframe=train_df,
    audio_dir=train_audio_dir,
    label2id=label2id,
)


In [ ]:
dev_dataset = AudioDataset(
    dataframe=dev_df,
    audio_dir=dev_audio_dir,
    label2id=label2id,
)

test_dataset = AudioDataset(
    dataframe=test_df_audio,
    audio_dir=test_audio_dir,
    label2id=label2id,
)


In [ ]:
print(len(train_dataset))
print(len(dev_dataset))
print(len(test_dataset))

9988
1108
2608


## 5. DataLoader

In [ ]:
from torch.utils.data import DataLoader

audio_collator = AudioCollator(
    feature_extractor=feature_extractor,
    sampling_rate=16000,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=audio_collator,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=audio_collator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=audio_collator,
)

## 6. Modeling



In [ ]:
#mean pooling
import torch.nn as nn


class AudioEmotionClassifier(nn.Module):
    def __init__(
        self,
        model_name="microsoft/wavlm-base-plus",
        num_classes=7,
        dropout=0.1,
    ):
        super().__init__()

        self.encoder = WavLMModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size
        # WavLM Base+ → 768

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )
        # 여기에선 768개의 특징을 받아서 7개의 감정 점수로 변환하는 역할

    def forward(
        self,
        input_values,
        attention_mask
    ):
        outputs = self.encoder(
            input_values=input_values,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state
        # [B, T, 768]

        feature_mask = self.encoder._get_feature_vector_attention_mask(
            hidden_states.shape[1],
            attention_mask
        )
        # [B, T] 마스크고

        mask = feature_mask.unsqueeze(-1).to(hidden_states.dtype)
        # [B, T, 1] 차원추가하는것

        summed = (hidden_states * mask).sum(dim=1)
        # [B, 768]

        counts = mask.sum(dim=1).clamp(min=1)
        # [B, 1]

        audio_embedding = summed / counts
        # [B, 768]

        audio_embedding = self.dropout(
            audio_embedding
        )
        # [B, 768]

        logits = self.classifier(
            audio_embedding
        )

        return logits

In [ ]:
model = AudioEmotionClassifier(
    model_name="microsoft/wavlm-base-plus",
    num_classes=7,
    dropout=0.1,
)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

In [ ]:
for param in model.encoder.feature_extractor.parameters():
    param.requires_grad = False

In [ ]:
model.encoder.config.apply_spec_augment = False

In [ ]:
model.encoder.gradient_checkpointing_enable()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

device: cuda
GPU: NVIDIA GeForce RTX 5090


## 7. Class Weights, Loss, Optimizer

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.arange(7)

train_labels = (
    train_df["Emotion"]
    .map(label2id)
    .values
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

print(class_weights)

for i, weight in enumerate(class_weights):
    print(
        f"{id2label[i]:8s}: {weight:.4f}"
    )

[1.286616   5.26515551 5.32409382 0.81862142 0.3030064  2.0891027
 1.18411381]
anger   : 1.2866
disgust : 5.2652
fear    : 5.3241
joy     : 0.8186
neutral : 0.3030
sadness : 2.0891
surprise: 1.1841


In [ ]:
import torch.nn as nn
from torch.optim import AdamW

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device,
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

optimizer = AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-5,
)

/tmp/ipykernel_13482/3914773551.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_weights_tensor = torch.tensor(


## 8. Mixed Precision과 학습·평가 함수

In [ ]:
use_amp = device.type == "cuda"
use_bf16 = use_amp and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

print("AMP enabled:", use_amp)
print("AMP dtype  :", amp_dtype)


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    accumulation_steps=4,
):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(loader, desc="Training")

    for step, batch in enumerate(progress):
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = model(
                input_values=input_values,
                attention_mask=attention_mask,
            )
            loss = criterion(logits, labels)
            scaled_loss = loss / accumulation_steps

        scaled_loss.backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        max_seconds = input_values.shape[1] / 16000
        progress.set_postfix(loss=f"{loss.item():.4f}", sec=f"{max_seconds:.1f}")

    return total_loss / len(loader)


In [ ]:
from sklearn.metrics import f1_score


def evaluate(
    model,
    loader,
    device,
):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc="Evaluating"
        ):

            input_values = batch[
                "input_values"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            labels = batch[
                "labels"
            ].to(device)

            logits = model(
                input_values=input_values,
                attention_mask=attention_mask,
            )

            preds = logits.argmax(dim=1)

            all_preds.extend(
                preds.cpu().tolist()
            )

            all_labels.extend(
                labels.cpu().tolist()
            )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        macro_f1,
        all_labels,
        all_preds,
    )

## 9. Checkpoint와 Main Training

기본 훈련은 3 epoch이며 dev macro F1이 가장 높은 모델만 저장합니다.

In [ ]:
from pathlib import Path

checkpoint_dir = Path(
    "/workspace/emotion-ai/checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_path = (
    checkpoint_dir
    / "wavlm_audio_baseline_best.pt"
)

In [ ]:
RESUME_FROM_CHECKPOINT = False

if RESUME_FROM_CHECKPOINT and checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print("loaded epoch:", checkpoint["epoch"])
    print("loaded dev F1:", checkpoint["dev_macro_f1"])
else:
    print("Starting a new training run.")


In [ ]:
NUM_EPOCHS = 3
best_dev_f1 = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n========== EPOCH {epoch}/{NUM_EPOCHS} ==========")

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
    )

    dev_f1, _, dev_pred = evaluate(
        model=model,
        loader=dev_loader,
        device=device,
    )

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Dev Macro F1: {dev_f1:.4f}")

    pred_counter = Counter(dev_pred)
    print("Prediction distribution:")
    for class_id in range(7):
        print(f"{id2label[class_id]:8s}: {pred_counter.get(class_id, 0)}")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "dev_macro_f1": dev_f1,
            },
            checkpoint_path,
        )
        print("✓ Best checkpoint saved")


## 10. Dev/Test 평가와 예측 저장

In [64]:
# best checkpoint를 불러와 dev 예측과 분포를 다시 계산합니다.
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

print("Best epoch:", checkpoint["epoch"])
print("Saved Dev Macro F1:", checkpoint["dev_macro_f1"])

model.load_state_dict(
    checkpoint["model_state_dict"]
)

dev_f1, y_true, y_pred = evaluate(
    model=model,
    loader=dev_loader,
    device=device,
)

from collections import Counter
from sklearn.metrics import classification_report

pred_counter = Counter(y_pred)

print(f"\nDev Macro F1: {dev_f1:.4f}")

print("\nPrediction distribution:")
for class_id in range(7):
    print(
        f"{id2label[class_id]:8s}: "
        f"{pred_counter.get(class_id, 0)}"
    )

Best epoch: 2
Saved Dev Macro F1: 0.30411478513572554


Evaluating:   0%|          | 0/554 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(



Dev Macro F1: 0.3041

Prediction distribution:
anger   : 197
disgust : 0
fear    : 0
joy     : 64
neutral : 604
sadness : 111
surprise: 132


In [65]:
# dev set의 클래스별 precision, recall, F1을 출력합니다.
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(7)),
        target_names=[
            id2label[i]
            for i in range(7)
        ],
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

       anger     0.4264    0.5490    0.4800       153
     disgust     0.0000    0.0000    0.0000        22
        fear     0.0000    0.0000    0.0000        40
         joy     0.4062    0.1595    0.2291       163
     neutral     0.5778    0.7441    0.6505       469
     sadness     0.3153    0.3153    0.3153       111
    surprise     0.4848    0.4267    0.4539       150

    accuracy                         0.5036      1108
   macro avg     0.3158    0.3135    0.3041      1108
weighted avg     0.4605    0.5036    0.4684      1108



In [67]:
# best checkpoint를 test set에서 평가합니다.
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_f1, test_y_true, test_y_pred = evaluate(
    model=model,
    loader=test_loader,
    device=device,
)

print(f"Test Macro F1: {test_f1:.4f}")

print(
    classification_report(
        test_y_true,
        test_y_pred,
        labels=list(range(7)),
        target_names=[
            id2label[i]
            for i in range(7)
        ],
        digits=4,
        zero_division=0,
    )
)

test_pred_counter = Counter(test_y_pred)

for class_id in range(7):
    print(
        f"{id2label[class_id]:8s}: "
        f"{test_pred_counter.get(class_id, 0)}"
    )

Evaluating:   0%|          | 0/1304 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Test Macro F1: 0.2882
              precision    recall  f1-score   support

       anger     0.4014    0.5130    0.4504       345
     disgust     0.0000    0.0000    0.0000        67
        fear     0.0000    0.0000    0.0000        50
         joy     0.3974    0.1496    0.2174       401
     neutral     0.6204    0.7548    0.6810      1256
     sadness     0.2797    0.3173    0.2973       208
    surprise     0.3929    0.3523    0.3715       281

    accuracy                         0.5176      2608
   macro avg     0.2988    0.2982    0.2882      2608
weighted avg     0.4776    0.5176    0.4847      2608

anger   : 441
disgust : 0
fear    : 0
joy     : 151
neutral : 1528
sadness : 236
surprise: 252


In [ ]:
# 오디오 모델의 샘플별 test 예측을 CSV로 저장합니다.
audio_result = test_df_audio[
    [
        "Dialogue_ID",
        "Utterance_ID",
        "Utterance",
        "Emotion",
    ]
].copy()

audio_result["true_id"] = test_y_true
audio_result["audio_pred_id"] = test_y_pred

results_dir = Path("/workspace/emotion-ai/results")
results_dir.mkdir(parents=True, exist_ok=True)
audio_result.to_csv(
    results_dir / "audio_test_predictions.csv",
    index=False,
)


## 11. 텍스트·오디오 결과 비교

In [72]:
# 텍스트와 오디오 모델의 저장된 test 예측을 읽습니다.
import pandas as pd

text_result = pd.read_csv(
    "/workspace/emotion-ai/results/text_test_predictions.csv"
)

audio_result = pd.read_csv(
    "/workspace/emotion-ai/results/audio_test_predictions.csv"
)

In [73]:
# 동일한 발화 키를 기준으로 두 모델의 예측을 합칩니다.
compare_df = text_result.merge(
    audio_result[
        [
            "Dialogue_ID",
            "Utterance_ID",
            "audio_pred_id",
        ]
    ],
    on=[
        "Dialogue_ID",
        "Utterance_ID",
    ],
    how="inner",
    validate="one_to_one",
)

print(compare_df.shape)

(2608, 7)


In [74]:
# 각 샘플에서 텍스트·오디오 모델의 정답 여부를 표시합니다.
compare_df["text_correct"] = (
    compare_df["text_pred_id"]
    == compare_df["true_id"]
)

compare_df["audio_correct"] = (
    compare_df["audio_pred_id"]
    == compare_df["true_id"]
)

In [75]:
# 두 모델이 맞고 틀린 조합별 샘플 수와 비율을 계산합니다.
both_correct = (
    compare_df["text_correct"]
    & compare_df["audio_correct"]
)

text_only = (
    compare_df["text_correct"]
    & ~compare_df["audio_correct"]
)

audio_only = (
    ~compare_df["text_correct"]
    & compare_df["audio_correct"]
)

both_wrong = (
    ~compare_df["text_correct"]
    & ~compare_df["audio_correct"]
)

n = len(compare_df)

print(
    "Both correct      :",
    both_correct.sum(),
    f"({both_correct.mean()*100:.1f}%)"
)

print(
    "Text only correct :",
    text_only.sum(),
    f"({text_only.mean()*100:.1f}%)"
)

print(
    "Audio only correct:",
    audio_only.sum(),
    f"({audio_only.mean()*100:.1f}%)"
)

print(
    "Both wrong        :",
    both_wrong.sum(),
    f"({both_wrong.mean()*100:.1f}%)"
)

Both correct      : 1065 (40.8%)
Text only correct : 610 (23.4%)
Audio only correct: 285 (10.9%)
Both wrong        : 648 (24.8%)


In [76]:
# 텍스트는 틀리고 오디오는 맞힌 사례를 일부 확인합니다.
audio_rescues = compare_df[audio_only].copy()

audio_rescues["text_pred"] = (
    audio_rescues["text_pred_id"].map(id2label)
)

audio_rescues["audio_pred"] = (
    audio_rescues["audio_pred_id"].map(id2label)
)

print(
    audio_rescues[
        [
            "Utterance",
            "Emotion",
            "text_pred",
            "audio_pred",
        ]
    ]
    .head(30)
    .to_string(index=False)
)

                                                                                                       Utterance  Emotion text_pred audio_pred
                                                                    Oh okay, lift it straight up over your head!    anger       joy      anger
                                                                                Yeah. I really like his glasses.  neutral       joy    neutral
                                                                             It kicked! I think the baby kicked! surprise     anger   surprise
                                                          Oh no wait, oh no, the elastic on my underwear busted.  neutral   sadness    neutral
                                                                                            What’s fish hooking? surprise   neutral   surprise
     Huh, what’s fish hooking...   Thanks man, that would have been really hard to describe. What is that taste?  neutral  surprise    neutral

## A1. 원본 오디오 한 파일 확인

In [1]:
# 원본 오디오 한 파일을 읽어 파형, sample rate, 길이를 확인합니다.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import soundfile as sf

audio_dir = Path(
    "/workspace/emotion-ai/data/meld_audio/extracted/train"
)

audio_files = sorted(audio_dir.glob("*.flac"))

print("train audio count:", len(audio_files))
print("first file:", audio_files[0])

waveform, sample_rate = sf.read(audio_files[0])

print("waveform type:", type(waveform))
print("waveform shape:", waveform.shape)
print("sample rate:", sample_rate)
print("dtype:", waveform.dtype)

print("\nfirst 10 values:")
print(waveform[:10])

train audio count: 9988
first file: /workspace/emotion-ai/data/meld_audio/extracted/train/dia0_utt0.flac
waveform type: <class 'numpy.ndarray'>
waveform shape: (90795,)
sample rate: 16000
dtype: float64

first 10 values:
[-0.00296116 -0.02849483 -0.00333774 -0.01904011 -0.05259562 -0.02441645
 -0.02068996 -0.0093801  -0.02941537  0.02109158]


In [6]:
# 샘플 수와 sample rate로 오디오 길이를 계산합니다.
duration = len(waveform) / sample_rate

print(f"duration: {duration:.2f} sec")

duration: 5.67 sec


## A2. Feature extractor와 WavLM 출력 확인

In [2]:
# WavLM feature extractor가 파형을 입력 텐서로 바꾸는 과정을 확인합니다.
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained(
    "microsoft/wavlm-base-plus"
)

inputs = feature_extractor(
    waveform,
    sampling_rate=sample_rate,
    return_tensors="pt"
)

print("keys:", inputs.keys())

print("\ninput_values")
print("shape:", inputs["input_values"].shape)
print("dtype:", inputs["input_values"].dtype)

print("\nattention_mask")
print("shape:", inputs["attention_mask"].shape)
print("dtype:", inputs["attention_mask"].dtype)

print("\nfirst 10 input values")
print(inputs["input_values"][0, :10])

print("\nattention mask first 20")
print(inputs["attention_mask"][0, :20])

keys: KeysView({'input_values': tensor([[-2.9612e-03, -2.8495e-02, -3.3377e-03,  ...,  4.0650e-05,
         -1.1826e-04, -4.8757e-05]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32)})

input_values
shape: torch.Size([1, 90795])
dtype: torch.float32

attention_mask
shape: torch.Size([1, 90795])
dtype: torch.int32

first 10 input values
tensor([-0.0030, -0.0285, -0.0033, -0.0190, -0.0526, -0.0244, -0.0207, -0.0094,
        -0.0294,  0.0211])

attention mask first 20
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       dtype=torch.int32)


In [3]:
# 한 오디오를 WavLM encoder에 통과시켜 hidden state를 확인합니다.
from transformers import WavLMModel
import torch

wavlm = WavLMModel.from_pretrained(
    "microsoft/wavlm-base-plus"
)

wavlm.eval()

with torch.no_grad():
    outputs = wavlm(
        input_values=inputs["input_values"],
        attention_mask=inputs["attention_mask"]
    )

hidden_states = outputs.last_hidden_state

print("outputs type:")
print(type(outputs))

print("\nlast_hidden_state:")
print(hidden_states.shape)
print(hidden_states.dtype)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


outputs type:
<class 'transformers.modeling_outputs.Wav2Vec2BaseModelOutput'>

last_hidden_state:
torch.Size([1, 283, 768])
torch.float32


In [11]:
# 시간축 hidden state를 단순 평균해 오디오 임베딩을 확인합니다.
audio_embedding = hidden_states.mean(dim=1)

print(audio_embedding.shape)
print(audio_embedding[0, :10])

torch.Size([1, 768])
tensor([-0.0480,  0.0316, -0.0394,  0.0061,  0.0049,  0.0134, -0.0102, -0.0182,
        -0.0026, -0.0048])


In [12]:
# padding을 제외하기 위한 downsampled attention mask를 확인합니다.
feature_mask = wavlm._get_feature_vector_attention_mask(
    hidden_states.shape[1],
    inputs["attention_mask"]
)

print(feature_mask.shape)
print(feature_mask.dtype)
print(feature_mask[0, :20])
print(feature_mask.sum())

torch.Size([1, 283])
torch.bool
tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True])
tensor(283)


In [ ]:
# 한 샘플을 별도의 분류기에 통과시켜 7개 감정 logits을 확인합니다.
walkthrough_classifier = AudioEmotionClassifier()
walkthrough_classifier.eval()

with torch.no_grad():
    walkthrough_logits = walkthrough_classifier(
        input_values=inputs["input_values"],
        attention_mask=inputs["attention_mask"],
    )

print("logits shape:", walkthrough_logits.shape)
print("logits:", walkthrough_logits)


## A3. CSV와 오디오 파일 무결성 점검

In [ ]:
# train CSV와 실제 오디오 파일 수를 비교할 audit 데이터를 읽습니다.
audit_train_df = pd.read_csv(data_dir / "train.csv")

print("CSV rows:", len(audit_train_df))
print("Audio files:", len(list(train_audio_dir.glob("*.flac"))))
print("CSV columns:", audit_train_df.columns.tolist())


In [ ]:
# 각 CSV 행에서 기대되는 오디오 파일명을 만듭니다.
audit_train_df["expected_audio"] = (
    "dia"
    + audit_train_df["Dialogue_ID"].astype(str)
    + "_utt"
    + audit_train_df["Utterance_ID"].astype(str)
    + ".flac"
)

print(
    audit_train_df[
        ["Dialogue_ID", "Utterance_ID", "Emotion", "expected_audio"]
    ].head()
)


In [ ]:
# 기대 오디오 파일이 실제 폴더에 존재하는지 검사합니다.
audit_train_df["audio_exists"] = audit_train_df["expected_audio"].apply(
    lambda name: (train_audio_dir / name).exists()
)

print("Matched audio:", audit_train_df["audio_exists"].sum())
print("Missing audio:", (~audit_train_df["audio_exists"]).sum())


In [ ]:
# 누락된 오디오에 해당하는 CSV 행을 확인합니다.
missing = audit_train_df.loc[
    ~audit_train_df["audio_exists"],
    ["Dialogue_ID", "Utterance_ID", "Emotion", "expected_audio"],
]

print(missing)


In [ ]:
# CSV에 없는 여분의 오디오 파일을 찾습니다.
actual_audio = {path.name for path in train_audio_dir.glob("*.flac")}
expected_audio = set(audit_train_df["expected_audio"])
extra_audio = actual_audio - expected_audio

print("Extra audio files:", len(extra_audio))
for name in sorted(extra_audio)[:20]:
    print(name)


In [ ]:
# Dialogue_ID와 Utterance_ID가 중복된 행을 찾습니다.
duplicate_mask = audit_train_df.duplicated(
    subset=["Dialogue_ID", "Utterance_ID"],
    keep=False,
)
duplicates = audit_train_df.loc[
    duplicate_mask,
    ["Dialogue_ID", "Utterance_ID", "Utterance", "Emotion"],
]

print("Duplicate rows:", len(duplicates))
if len(duplicates) > 0:
    print(duplicates)


In [ ]:
# 오디오가 존재하는 audit 행만 남깁니다.
audit_train_df = audit_train_df[
    audit_train_df["audio_exists"]
].reset_index(drop=True)

print("filtered audit rows:", len(audit_train_df))


In [41]:
# CSV와 폴더 사이의 누락·여분 오디오를 검사하는 함수를 정의합니다.
def check_audio_integrity(df, audio_dir):

    audio_dir = Path(audio_dir)

    expected = {
        f"dia{row.Dialogue_ID}_utt{row.Utterance_ID}.flac"
        for _, row in df.iterrows()
    }

    actual = {
        path.name
        for path in audio_dir.glob("*.flac")
    }

    missing = expected - actual
    extra = actual - expected

    print("CSV rows :", len(df))
    print("Audio    :", len(actual))
    print("Missing  :", len(missing))
    print("Extra    :", len(extra))

    if missing:
        print("\nMissing examples:")
        print(sorted(missing)[:10])

    if extra:
        print("\nExtra examples:")
        print(sorted(extra)[:10])

    return missing, extra


In [24]:
# dev/test CSV와 오디오 폴더의 파일 구성을 점검합니다.
check_audio_integrity(
    dev_df,
    "/workspace/emotion-ai/data/meld_audio/extracted/dev"
)

check_audio_integrity(
    test_df,
    "/workspace/emotion-ai/data/meld_audio/extracted/test"
)

NameError: name 'check_audio_integrity' is not defined

## A4. 오디오 길이와 이상치 점검

In [26]:
# 각 오디오 파일의 재생 길이를 빠르게 읽는 함수를 정의합니다.
import soundfile as sf
import pandas as pd


def get_audio_durations(df, audio_dir):

    audio_dir = Path(audio_dir)

    durations = []

    for _, row in df.iterrows():

        audio_path = (
            audio_dir
            / f"dia{row.Dialogue_ID}_utt{row.Utterance_ID}.flac"
        )

        info = sf.info(audio_path)

        duration = info.frames / info.samplerate

        durations.append(duration)

    return pd.Series(durations)

In [27]:
# train/dev/test 오디오 길이의 요약 통계를 계산합니다.
train_durations = get_audio_durations(
    train_df,
    "/workspace/emotion-ai/data/meld_audio/extracted/train"
)

dev_durations = get_audio_durations(
    dev_df,
    "/workspace/emotion-ai/data/meld_audio/extracted/dev"
)

test_durations = get_audio_durations(
    test_df,
    "/workspace/emotion-ai/data/meld_audio/extracted/test"
)

for name, durations in [
    ("TRAIN", train_durations),
    ("DEV", dev_durations),
    ("TEST", test_durations),
]:
    print("\n", "=" * 20, name, "=" * 20)

    print(
        durations.describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )


 ==================== TRAIN ====================
count    9988.000000
mean        3.142124
std         2.428665
min         0.064000
50%         2.474687
90%         6.357312
95%         7.936000
99%        11.781549
max        41.045313
dtype: float64

 ==================== DEV ====================
count    1108.000000
mean        3.122638
std         2.498736
min         0.064000
50%         2.442656
90%         5.986119
95%         7.607453
99%        11.907859
max        28.544000
dtype: float64

 ==================== TEST ====================
count    2610.000000
mean        3.396847
std         7.791957
min         0.128000
50%         2.602688
90%         6.272000
95%         7.765313
99%        11.392000
max       304.960000
dtype: float64


In [28]:
# 지정한 길이보다 긴 오디오 샘플을 찾는 함수를 정의합니다.
def show_duration_outliers(
    df,
    durations,
    audio_dir,
    threshold=15.0,
):
    outlier_idx = durations[
        durations > threshold
    ].index

    rows = df.loc[
        outlier_idx,
        [
            "Dialogue_ID",
            "Utterance_ID",
            "Utterance",
            "Emotion",
        ],
    ].copy()

    rows["duration"] = durations.loc[outlier_idx].values

    rows["audio_file"] = rows.apply(
        lambda row:
        f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.flac",
        axis=1
    )

    return rows.sort_values(
        "duration",
        ascending=False
    )

In [29]:
# test에서 15초를 넘는 오디오를 확인합니다.
test_outliers = show_duration_outliers(
    test_df,
    test_durations,
    "/workspace/emotion-ai/data/meld_audio/extracted/test",
    threshold=15.0,
)

print(test_outliers.head(20).to_string())

      Dialogue_ID  Utterance_ID                                                                                                                                                Utterance  Emotion    duration        audio_file
348            38             4                                                                                                                            Oh its great, its a role on      joy  304.960000   dia38_utt4.flac
2052          220             0                                                                                                                                       Whats that smell?  disgust  235.072000  dia220_utt0.flac
782            90             1                                                                                                                She is not
  Shes gay. Oh my God. She is    anger   16.725313   dia90_utt1.flac
413            46             8                              Sticky shoes, sticky shoes, always make me

In [30]:
# train/dev에서 15초를 넘는 오디오를 확인합니다.
train_outliers = show_duration_outliers(
    train_df,
    train_durations,
    "/workspace/emotion-ai/data/meld_audio/extracted/train",
    threshold=15.0,
)

dev_outliers = show_duration_outliers(
    dev_df,
    dev_durations,
    "/workspace/emotion-ai/data/meld_audio/extracted/dev",
    threshold=15.0,
)

print(train_outliers.head(20).to_string())
print(dev_outliers.head(20).to_string())

      Dialogue_ID  Utterance_ID                                                                                                                                                                                                                                          Utterance   Emotion   duration         audio_file
2922          309             0                                                                                                                              Kathy! Kathy! Hi!! Kathy! Kathy! (She doesn't hear him and keeps running, Chandler starts chasing her as the theme to       joy  41.045313   dia309_utt0.flac
4543          485             0                                                                                                                                                                                                                                Hey, this-  Heyy...       joy  25.792000   dia485_utt0.flac
7142          757             2  Y'know, I don't know i

In [56]:
# 문제가 의심되는 test 오디오의 앞 10초를 직접 들어봅니다.
import soundfile as sf
from IPython.display import Audio, display

for filename in [
    "dia38_utt4.flac",
    "dia220_utt0.flac",
]:
    path = test_audio_dir / filename

    waveform, sr = sf.read(path, dtype="float32")

    first_10_sec = waveform[: sr * 10]

    print(filename, len(waveform) / sr)
    display(Audio(first_10_sec, rate=sr))

dia38_utt4.flac 304.96


dia220_utt0.flac 235.072


## A5. Dataset과 batch 샘플 확인

In [17]:
# train Dataset의 전체 샘플 수를 확인합니다.
print("dataset size:", len(train_dataset))

dataset size: 9988


In [28]:
# Dataset 한 샘플의 파형, 라벨, 파일 경로를 확인합니다.
sample = train_dataset[0]

print(sample.keys())

print("waveform shape:", sample["waveform"].shape)
print("waveform dtype:", sample["waveform"].dtype)

print("label:", sample["label"])
print("emotion:", id2label[sample["label"].item()])

print("audio path:", sample["audio_path"])

dict_keys(['waveform', 'label', 'audio_path'])
waveform shape: torch.Size([90795])
waveform dtype: torch.float32
label: tensor(4)
emotion: neutral
audio path: /workspace/emotion-ai/data/meld_audio/extracted/train/dia0_utt0.flac


In [ ]:
# 배치 동작 확인용 batch-size 1 DataLoader를 만듭니다.
debug_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=audio_collator,
)


In [ ]:
# 첫 미니배치의 입력, mask, 라벨, 파일 경로를 확인합니다.
batch = next(iter(debug_loader))

print("input_values:")
print(batch["input_values"].shape)
print(batch["input_values"].dtype)

print("\nattention_mask:")
print(batch["attention_mask"].shape)
print(batch["attention_mask"].dtype)

print("\nlabels:")
print(batch["labels"])
print(batch["labels"].shape)

print("\naudio paths:")
for path in batch["audio_paths"]:
    print(path)


In [32]:
# padding된 길이와 각 샘플의 실제 길이를 비교합니다.
print("padded length:")
print(batch["input_values"].shape[1])

print("\nreal sample lengths:")
print(batch["attention_mask"].sum(dim=1))

padded length:
52907

real sample lengths:
tensor([23211, 18773, 52907, 45397])
